In [ ]:
# yaaihh burst worker: run this one cell. Colab secrets (the key icon): RCLONE_CONFIG (the `rclone config show colab`
# section from the tower: a drive.file remote rooted at the yaaihh-colab folder, README step 1); for Layer 2 also
# COLAB_RPC_URL (https://...) and COLAB_RPC_TOKEN. Work runs on /content/work.
import hashlib, io, json, os, pathlib, subprocess, sys, urllib.request, zipfile
from google.colab import userdata

REMOTE = "colab:"  # the remote is rooted at the folder submit.py fills
RCLONE = ("v1.76.0", "b00f00b3266c3764413afe599d3ec316f6f43b3e0ea8a0bc1d186d2133e46df7")  # release, its zip's sha256
WITH_BUILDAH = False  # True for jobs submitted with --oci
W = pathlib.Path("/content/work")
(W / "colab").mkdir(parents=True, exist_ok=True)


def secret(key):
    try:
        return userdata.get(key)
    except Exception:  # not set, or this notebook not given it
        return None


def private_file(path, text):
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with os.fdopen(os.open(p, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600), "w") as f:
        f.write(text)
    return p


subprocess.run(["apt-get", "-qq", "update"], check=True, stdout=subprocess.DEVNULL)
subprocess.run(["apt-get", "-qq", "install", "-y", "zstd"] + (["buildah"] if WITH_BUILDAH else []),
               check=True, stdout=subprocess.DEVNULL)
v, want = RCLONE
z = urllib.request.urlopen(f"https://downloads.rclone.org/{v}/rclone-{v}-linux-amd64.zip", timeout=120).read()
assert hashlib.sha256(z).hexdigest() == want, "the rclone download is not the pinned release"
(W / "bin").mkdir(exist_ok=True)
(W / "bin" / "rclone").write_bytes(zipfile.ZipFile(io.BytesIO(z)).read(f"rclone-{v}-linux-amd64/rclone"))
(W / "bin" / "rclone").chmod(0o755)
env = {**os.environ, "PATH": f"{W / 'bin'}:{os.environ['PATH']}"}
private_file("/root/.config/rclone/rclone.conf", secret("RCLONE_CONFIG") or "")
subprocess.run(["rclone", "copy", f"{REMOTE}worker", str(W / "colab")], check=True, env=env)
for f, h in json.loads((W / "colab" / "MANIFEST.json").read_text()).items():
    assert hashlib.sha256((W / "colab" / f).read_bytes()).hexdigest() == h, f"{f}: not the manifest's"
cmd = [sys.executable, "-I", str(W / "colab" / "worker.py"), "--store", REMOTE, "--work", str(W)]
url, token = secret("COLAB_RPC_URL"), secret("COLAB_RPC_TOKEN")
if url and token:
    cmd += ["--rpc", url, "--token-file", str(private_file("/root/.colab-rpc-token", token))]
subprocess.run(cmd, env=env)
